# ⚙️ 特征工程

本Notebook用于特征工程和特征选择

In [ ]:
# 导入必要的库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 导入自定义模块
import sys
sys.path.append('../src')
from data_processing import load_data
from feature_engineering import create_derived_features, encode_categorical_features, normalize_numeric_features, select_features, identify_feature_types, feature_engineering_pipeline

In [ ]:
# 加载清洗后的数据
df = load_data('../data/processed/cleaned_ev_data.csv')
print(f"数据形状: {df.shape}")

In [ ]:
# 识别特征类型
feature_types = identify_feature_types(df)

print(f"数值特征: {feature_types['numeric_cols']}")
print(f"\n分类特征: {feature_types['categorical_cols']}")
print(f"\nAI特征: {feature_types['ai_feature_cols']}")

In [ ]:
# 创建派生特征
df_engineered = create_derived_features(df)
print(f"添加派生特征后的数据形状: {df_engineered.shape}")
print(f"\n新增特征: {set(df_engineered.columns) - set(df.columns)}")

In [ ]:
# 查看新增的派生特征
df_engineered[['car_age', 'price_range', 'range_efficiency', 'generation']].head()

In [ ]:
# 编码分类特征
categorical_cols = ['Make', 'Model', 'Electric Vehicle Type', 'price_range', 'generation']
df_encoded = encode_categorical_features(df_engineered, categorical_cols, method='label')
print(f"编码后的数据形状: {df_encoded.shape}")

In [ ]:
# 标准化数值特征
numeric_cols = ['Base MSRP', 'Electric Range', 'car_age', 'range_efficiency']
df_normalized, scaler = normalize_numeric_features(df_encoded, numeric_cols)
print(f"标准化后的统计描述:\n{df_normalized[numeric_cols].describe().round(2)}")

In [ ]:
# 完整的特征工程流程
result = feature_engineering_pipeline(df, target_col='Base MSRP')

X = result['X']
y = result['y']

print(f"特征矩阵形状: {X.shape}")
print(f"目标变量形状: {y.shape}")
print(f"\n特征名称:\n{X.columns.tolist()}")

In [ ]:
# 特征选择
selection_result = select_features(X, y, k=15, method='f_regression')

print("特征重要性排名:")
print(selection_result['feature_importance'])

In [ ]:
# 可视化特征重要性
plt.figure(figsize=(12, 8))
sns.barplot(data=selection_result['feature_importance'], x='score', y='feature', palette='viridis')
plt.title('特征重要性排名')
plt.xlabel('F统计量')
plt.ylabel('特征')
plt.show()

In [ ]:
# 保存特征工程后的数据集
df_final = pd.concat([X, y], axis=1)
df_final.to_csv('../data/processed/engineered_features.csv', index=False)
print("特征工程后的数据已保存到 '../data/processed/engineered_features.csv'")

## 📝 特征工程总结

1. **派生特征**: 创建了车龄、价格区间、续航效率、年代分组等特征
2. **分类编码**: 使用标签编码处理分类变量
3. **数值标准化**: 使用StandardScaler标准化数值特征
4. **特征选择**: 使用F统计量选择了15个最重要的特征
5. **最终特征**: 特征矩阵包含XX个特征，目标变量为价格